# 01 Data Preprocessing
## Purpose
Prepare spatial transcriptomics (GSE248904) and AD GWAS (Bellenguez 2022) data for gsMap analysis.

**Paper position**: Supports all downstream analyses; QC metrics feed into **Figure 1** and **Supplementary Figure S1**.

### Steps:
1. Load and inspect whole-mouse spatial transcriptomics data
2. Filter to control (untreated) samples only
3. Add spatial coordinates to `obsm["spatial"]` and raw counts to `layers["count"]`
4. Split by organ for per-organ gsMap analysis
5. Format AD GWAS summary statistics for gsMap

In [1]:
%matplotlib inline
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

BASE = '../analysis/26_gsmap'
DATA = f'{BASE}/data'
ST_PATH = f'{DATA}/st/GSE248904_All_Samples.h5ad'
GWAS_PATH = f'{DATA}/gwas/Bellenguez2022_AD_GRCh38.tsv.gz'
RESOURCE_DIR = f'{DATA}/gsMap_resource'
OUTPUT_ST = f'{DATA}/st/per_organ'
os.makedirs(OUTPUT_ST, exist_ok=True)

print("Loading full dataset (backed mode)...")
adata = ad.read_h5ad(ST_PATH, backed='r')
print(f"Shape: {adata.shape}")
print(f"Samples: {adata.obs['Sample'].unique().tolist()}")
print(f"Organs: {adata.obs['Organ_Full_Name'].unique().tolist()}")
print(f"Treatments: {adata.obs['Treatment'].unique().tolist()}")

Loading full dataset (backed mode)...


Shape: (2336059, 32285)
Samples: ['CTRL_1', 'CTRL_2', 'LPS_1', 'LPS_2']
Organs: ['Muscle', 'Lung', 'Other', 'Brown Fat', 'Bone Marrow', 'Liver', 'Skin', 'Small Intestine', 'Brain', 'Colon', 'Kidney', 'Stomach', 'Heart', 'Spleen', 'Pancreas', 'Lymph Node', 'Thymus']
Treatments: ['Control', 'LPS']


In [2]:
# Overview of spot counts per organ and sample (control only)
ctrl_obs = adata.obs[adata.obs['Treatment'] == 'Control']
overview = ctrl_obs.groupby(['Sample', 'Organ_Full_Name']).size().unstack(fill_value=0)
print("Control samples - spots per organ:")
print(overview.to_string())
print(f"\nTotal control spots: {ctrl_obs.shape[0]}")
print(f"\nSubregions available: {ctrl_obs['Subregion'].dropna().unique().tolist()}")

Control samples - spots per organ:
Organ_Full_Name  Bone Marrow  Brain  Brown Fat  Colon  Heart  Kidney  Liver   Lung  Lymph Node  Muscle  Other  Pancreas   Skin  Small Intestine  Spleen  Stomach  Thymus
Sample                                                                                                                                                                  
CTRL_1                 21520  45312      12987  15714  13293   17469  41872  14979        2070  257133  36070      8570  53402            31853    4683     9809    1563
CTRL_2                 39364  61512      18470  23181   6604   14266  51176  18935        1361  194920  42564     15080  69925            28304    5145    11165    8717
LPS_1                      0      0          0      0      0       0      0      0           0       0      0         0      0                0       0        0       0
LPS_2                      0      0          0      0      0       0      0      0           0       0      0         0 


Subregions available: ['Muscle Tissue', 'Alveoli', 'Brown Fat Tissue', 'Bone Marrow Tissue', 'Pericentral', 'Other', 'Periportal', 'Cerebellar Cortex (Mol. Layer)', 'Colon Tissue', 'Cerebral Cortex', 'Small Intestine Tissue', 'Inner Stripe of Outer Medulla', 'Gastric Glands (Parietal Cell Rich)', 'Midbrain', 'Outer Stripe of Outer Medulla', 'Gastric Glands (Chief Cell Rich)', 'Renal Cortex', 'Hypodermis', 'Heart Tissue', 'Blood Vessel', 'Foveolar Epithelium', 'Caudoputamen', 'Dermis', 'White Pulp', 'Pancreas Tissue', 'Hyppocampus', 'Marginal Zone', 'Lymph Node Tissue', 'Bronchioles', 'Cerebellar Cortex (Gran. Layer)', 'Fiber Tracts', 'Red Pulp', 'Thymic Cortex', 'Ventricle', 'Inner Medulla', 'Epidermis', 'Midbrain/Hindbrain', 'Thymic Medulla', 'Spinal Cord', 'Thalamus', 'Meninges\xa0', 'Bronchi']


## Split control samples by organ and save as gsMap-compatible h5ad files
- Add `obsm["spatial"]` from `x_plotting` / `y_plotting`
- Store raw counts in `layers["count"]`
- Use `Subregion` as annotation (finer than Organ, better for gsMap)

In [3]:
# Split control data by organ and save gsMap-compatible h5ad files
# Load full data into memory for control samples only
print("Loading control samples into memory (this takes a while)...")
adata_full = ad.read_h5ad(ST_PATH)
adata_ctrl = adata_full[adata_full.obs['Treatment'] == 'Control'].copy()
del adata_full
import gc; gc.collect()
print(f"Control data shape: {adata_ctrl.shape}")

# Check if X contains raw counts
x_sample = adata_ctrl.X[:100].toarray() if hasattr(adata_ctrl.X, 'toarray') else adata_ctrl.X[:100]
is_counts = np.allclose(x_sample, np.round(x_sample)) and x_sample.min() >= 0
print(f"X appears to be raw counts: {is_counts}")
print(f"X dtype: {adata_ctrl.X.dtype}, min={x_sample.min():.2f}, max={x_sample.max():.2f}")

Loading control samples into memory (this takes a while)...


Control data shape: (1198988, 32285)
X appears to be raw counts: True
X dtype: float32, min=0.00, max=353.00


In [4]:
# Add spatial coordinates and save per-organ h5ad files
saved_files = {}
organs = adata_ctrl.obs['Organ_Full_Name'].unique()
# Exclude "Other" as it's not a defined organ
organs = [o for o in organs if o != 'Other']

for sample in ['CTRL_1', 'CTRL_2']:
    for organ in sorted(organs):
        mask = (adata_ctrl.obs['Sample'] == sample) & (adata_ctrl.obs['Organ_Full_Name'] == organ)
        adata_sub = adata_ctrl[mask].copy()
        
        if adata_sub.shape[0] < 100:
            print(f"  Skipping {organ}_{sample}: only {adata_sub.shape[0]} spots")
            continue
        
        # Add spatial coordinates to obsm
        adata_sub.obsm['spatial'] = adata_sub.obs[['x_plotting', 'y_plotting']].values.astype(np.float64)
        
        # Store raw counts in layers["count"]
        adata_sub.layers['count'] = adata_sub.X.copy()
        
        # Use Subregion as annotation; convert categorical to str first, then fill NaN
        subregion = adata_sub.obs['Subregion'].astype(str)
        subregion = subregion.replace('nan', organ)
        adata_sub.obs['annotation'] = subregion
        
        # Clean organ name for filename
        organ_clean = organ.replace(' ', '_')
        sample_clean = sample.replace('_', '')
        fname = f"{organ_clean}_{sample_clean}.h5ad"
        outpath = f"{OUTPUT_ST}/{fname}"
        adata_sub.write_h5ad(outpath)
        saved_files[f"{organ_clean}_{sample_clean}"] = outpath
        print(f"  Saved {fname}: {adata_sub.shape[0]} spots, {adata_sub.obs['annotation'].nunique()} annotations")

print(f"\nTotal files saved: {len(saved_files)}")
del adata_ctrl; gc.collect()

  Saved Bone_Marrow_CTRL1.h5ad: 21520 spots, 1 annotations


  Saved Brain_CTRL1.h5ad: 45312 spots, 8 annotations


  Saved Brown_Fat_CTRL1.h5ad: 12987 spots, 1 annotations


  Saved Colon_CTRL1.h5ad: 15714 spots, 1 annotations


  Saved Heart_CTRL1.h5ad: 13293 spots, 1 annotations


...

  Saved Skin_CTRL2.h5ad: 69925 spots, 4 annotations


  Saved Small_Intestine_CTRL2.h5ad: 28304 spots, 2 annotations
  Saved Spleen_CTRL2.h5ad: 5145 spots, 3 annotations


  Saved Stomach_CTRL2.h5ad: 11165 spots, 4 annotations


  Saved Thymus_CTRL2.h5ad: 8717 spots, 2 annotations

Total files saved: 32


3524

## Format AD GWAS summary statistics for gsMap
Convert Bellenguez et al. 2022 GWAS to gsMap format (SNP, A1, A2, Z, N).

In [5]:
import subprocess, shlex

# First, add N column (n_cases + n_controls) to GWAS file
gwas_with_n = f'{DATA}/gwas/Bellenguez2022_AD_withN.tsv.gz'
print("Adding N column to GWAS data...")
import gzip

chunksize = 500000
first_chunk = True
with gzip.open(gwas_with_n, 'wt') as fout:
    for chunk in pd.read_csv(GWAS_PATH, sep='\t', chunksize=chunksize):
        chunk['N'] = chunk['n_cases'] + chunk['n_controls']
        if first_chunk:
            chunk.to_csv(fout, sep='\t', index=False)
            first_chunk = False
        else:
            chunk.to_csv(fout, sep='\t', index=False, header=False)
print(f"Saved GWAS with N column: {gwas_with_n}")

# Now format with gsMap
gwas_out = f'{DATA}/gwas/AD_Bellenguez2022'
cmd = (
    f'../env/gsmap/bin/gsmap format_sumstats '
    f'--sumstats {gwas_with_n} '
    f'--snp variant_id '
    f'--a1 effect_allele '
    f'--a2 other_allele '
    f'--beta beta '
    f'--se standard_error '
    f'--p p_value '
    f'--frq effect_allele_frequency '
    f'--n N '
    f'--out {gwas_out}'
)
print(f"\nRunning gsMap format_sumstats...")
result = subprocess.run(shlex.split(cmd), capture_output=True, text=True, timeout=1200)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
else:
    outfile = f'{gwas_out}.sumstats.gz'
    with gzip.open(outfile, 'rt') as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 4: break
    # Count SNPs
    n_snps = sum(1 for _ in gzip.open(outfile, 'rt')) - 1
    print(f"\nGWAS formatted successfully: {outfile}")
    print(f"Total SNPs after QC: {n_snps:,}")

Adding N column to GWAS data...


Saved GWAS with N column: data/gwas/Bellenguez2022_AD_withN.tsv.gz

Running gsMap format_sumstats...


                                   ___  ___            
                                   |  \/  |            
                          __ _ ___ | .  . | __ _ _ __  
                         / _` / __|| |\/| |/ _` | '_ \ 
                        | (_| \__ \| |  | | (_| | |_) |
                         \__, |___/\_|  |_/\__,_| .__/ 
                          __/ |                 | |    
                         |___/                  |_|
                                Version: 1.73.7                                 
Using the following arguments for FormatSumstatsConfig:
{   'OR': None,
    'a1': 'effect_allele',
    'a2': 'other_allele',
    'beta': 'beta',
    'chr': 'Chr',
    'chunksize': 1000000.0,
    'dbsnp': None,
    'format': 'gsMap',
    'frq': 'effect_allele_frequency',
    'info': None,
    'info_min': 0.9,
    'keep_chr_pos': False,
    'maf_min': 0.01,
    'n': 'N',
    'out': 'data/gwas/AD_Bellenguez2022',
    'p': 'p_value',
    'pos': 'Pos',
    'se': 'standard_err


GWAS formatted successfully: data/gwas/AD_Bellenguez2022.sumstats.gz
Total SNPs after QC: 6,809,862
